In [5]:
from IPython.display import display, Math, Latex

import pandas as pd
import numpy as np
import numpy_financial as npf
import yfinance as yf
import matplotlib.pyplot as plt
import random
from datetime import datetime

Some global variables

In [6]:
filter_start = "2025-10-01"
filter_end = "2026-10-01"

period_start = "2023-10-01"
period_end = "2026-10-01"

In [7]:
# Set up exchange rate dataframe for every day in the period

exchange_rate = yf.Ticker("CAD=X").history(start = period_start, end = period_end)["Close"]

# Keep only year month date, datetime objects, in index column of dataframe
exchange_rate.index = pd.to_datetime(exchange_rate.index.strftime('%Y-%m-%d'))

display(exchange_rate.head())

Date
2023-10-02    1.35737
2023-10-03    1.36763
2023-10-04    1.37091
2023-10-05    1.37442
2023-10-06    1.37057
Name: Close, dtype: float64

Extracting stocks

In [8]:
# EXTRACTING TICKERS. MAKE SURE TO CHANGE FILE NAME FOR ACTUAL SUBMISSION
tickers = pd.read_csv("Tickers.csv", header=None)[0].to_list()
close_prices = pd.DataFrame()

for tick in tickers:
    t = yf.Ticker(tick)
    information = t.info

    # Check if there is data in here (i.e. not a delisted or invalid ticker)
    if t.history(period='max').empty:
        print(tick, "doesn't exist")
        continue
    
    # Check if it's a CAD or USD stock
    if (information["market"] != "ca_market") and (information["market"] != "us_market"):
        print(tick, "wrong market")
        continue

    # Check if it's a stock and not something like a bond or index fund
    if information["quoteType"] != "EQUITY":
        print(tick, "not stock")
        continue

    if (information["currency"] != "CAD") and (information["currency"] != "USD"):
        print(tick, "currency wrong")
        continue
    

    temp = yf.download(tick, filter_start, filter_end)

    if temp["Volume"].mean()[tick] < 5000:
        print(tick, "not enough volume")
        continue
    
    close_prices[tick] = temp["Close"]

# Filters out the first and last trading day of each month
idx = close_prices.index.to_series()
date_series_first=idx.groupby([idx.dt.year, idx.dt.month]).first()
date_series_last=idx.groupby([idx.dt.year, idx.dt.month]).last()
monthly_first = []
monthly_last = []
for i in date_series_first:
    monthly_first.append(i)
for i in date_series_last:
    monthly_last.append(i)

for i in range(len(monthly_first)):
    start_ind = close_prices.index.get_loc(monthly_first[i])
    end_ind = close_prices.index.get_loc(monthly_last[i])

    if (end_ind - start_ind + 1) < 18:
        close_prices.drop(close_prices.loc[monthly_first[i]:monthly_last[i]].index)

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
$AGN: possibly delisted; no timezone found


AGN doesn't exist


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
$BK: possibly delisted; no price data found  (1d 1927-12-20 -> 2026-11-25)


BK doesn't exist


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
$CELG: possibly delisted; no timezone found


CELG doesn't exist


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
$MON: possibly delisted; no timezone found


MON doesn't exist


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
$RTN: possibly delisted; no timezone found


RTN doesn't exist


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


In [9]:
display(close_prices.head())

,AAPL,ABBV,ABT,ACN,AIG,AMZN,AXP,BA,BAC,BB.TO,...,PYPL,QCOM,SHOP.TO,T.TO,TD.TO,TXN,UNH,UNP,UPS,USB
Date,,,,,,,,,,,,,,,,,,,,,
2025-10-01,254.316605,235.580246,129.656540,236.889038,77.725349,220.630005,324.080414,215.199997,49.622257,6.67,...,65.924812,163.093613,208.619995,20.261112,107.826416,174.982300,339.266174,229.894775,78.023796,46.106377
2025-10-02,255.989151,228.041824,129.190262,237.501389,77.725349,222.410004,325.974243,217.429993,49.426430,6.63,...,68.169777,165.405441,211.240005,19.976391,106.874405,176.854446,344.545624,229.679306,79.124153,46.135265
2025-10-03,256.875214,225.487259,130.744553,238.453979,79.072166,219.509995,325.737549,216.300003,49.583092,6.28,...,68.486244,165.728699,224.889999,20.104975,108.913048,174.914398,350.857574,231.912247,80.243011,46.289333
2025-10-06,255.551132,221.901215,129.918808,241.224197,81.024086,220.899994,327.463623,219.729996,49.338310,6.68,...,70.503754,165.180145,229.520004,19.893732,108.951500,176.359711,349.464630,232.509674,80.095062,45.952301
2025-10-07,255.342041,224.446136,129.219421,244.198547,82.029327,221.779999,323.498474,221.820007,49.240395,6.48,...,73.787140,162.084625,225.100006,19.875360,108.605331,171.742416,354.227814,227.074203,79.503273,45.740452


In [10]:
full_data = pd.DataFrame()

for tick in close_prices:
    prices = yf.download(tick, period_start, period_end)["Close"]
    
    if yf.Ticker(tick).info["currency"] == "CAD":
        full_data[tick] = prices[tick]
    else:
        full_data[tick] = prices[tick] * exchange_rate

full_data["CBIL.TO"] = yf.download("CBIL.TO", period_start, period_end, auto_adjust=True)["Close"]

full_data = full_data

display(full_data.head())

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%********

,AAPL,ABBV,ABT,ACN,AIG,AMZN,AXP,BA,BAC,BB.TO,...,QCOM,SHOP.TO,T.TO,TD.TO,TXN,UNH,UNP,UPS,USB,CBIL.TO
Date,,,,,,,,,,,,,,,,,,,,,
2023-10-02,232.519145,180.319180,121.325716,393.013051,76.241147,175.725132,195.350702,254.954813,33.728010,6.02,...,141.522611,73.839996,17.100691,70.040100,198.282192,655.916130,257.465521,174.719987,38.394417,45.243153
2023-10-03,232.456407,180.677267,122.319401,393.375974,75.013721,170.570816,190.994134,258.372658,32.977460,5.98,...,141.424401,70.440002,17.204720,69.378128,196.408622,654.324020,260.871062,174.530914,37.721729,45.229576
2023-10-04,234.716907,181.429954,122.433557,398.388435,76.116891,174.105576,193.537665,255.990027,33.094827,5.86,...,141.326160,72.230003,17.756870,69.195213,197.781115,657.373709,259.559025,174.915360,37.691544,45.243153
2023-10-05,237.011728,181.598940,123.452830,399.756958,77.327381,173.121948,195.441646,256.040701,33.333050,5.20,...,141.623504,71.510002,18.100962,69.997559,194.697560,666.297640,258.126533,174.975326,38.550101,45.247665
2023-10-06,239.833993,182.060462,123.977207,401.894122,78.033789,175.378129,197.848890,256.817403,33.252435,5.15,...,142.615890,73.269997,18.044947,70.359032,196.517888,675.474447,259.738334,175.497638,38.430053,45.256706


Sharpe calculation

In [11]:
half_life = 100

In [12]:
def get_stats(tick, half_life):
    s = full_data[tick].dropna()
    delta = np.log(1 + s.pct_change()).ewm(halflife = half_life)
    standard_deviation = delta.std().iloc[-1]
    average = delta.mean().iloc[-1]

    return average, standard_deviation

In [13]:
Rf = get_stats("CBIL.TO", half_life)[0]
print("Risk free return:", Rf)

sharpe = {}

for tick in close_prices:
    stats = get_stats(tick, half_life)
    sharpe[tick] = (stats[0] - Rf)/stats[1]

sharpe = dict(sorted(sharpe.items(), key = lambda item:item[1]))
print(sharpe)

Risk free return: 9.330034271459087e-05
{'T.TO': np.float64(-0.13693389765984107), 'PEP': np.float64(-0.05341844066465649), 'BA': np.float64(-0.03915028464493271), 'ABT': np.float64(-0.02650234699371945), 'UPS': np.float64(-0.022055935927177396), 'ACN': np.float64(-0.020406607626951446), 'AXP': np.float64(-0.0187092280275696), 'LMT': np.float64(-0.008226268662804882), 'PYPL': np.float64(-0.006419300331452179), 'AIG': np.float64(-0.005801933035182074), 'CL': np.float64(-0.005409532674453388), 'UNH': np.float64(0.004922993558513601), 'BLK': np.float64(0.01138565039508921), 'BAC': np.float64(0.017796636980047794), 'AMZN': np.float64(0.020084173228586952), 'QCOM': np.float64(0.02234483018855014), 'SHOP.TO': np.float64(0.032695244274573015), 'UNP': np.float64(0.03281646836889642), 'MO': np.float64(0.0335945850575999), 'USB': np.float64(0.03878268454027843), 'LLY': np.float64(0.04433440945341694), 'BMY': np.float64(0.04727078814127248), 'PM': np.float64(0.05075773824209209), 'TXN': np.float6

In [14]:
def weighted_corr(t1, t2):
    d1 = full_data[t1].pct_change(fill_method=None)
    d2 = full_data[t2].pct_change(fill_method=None)
    return d1.ewm(halflife=half_life).corr(d2).dropna().iloc[-1]

Correlation matrix

In [15]:
corr_mat = pd.DataFrame()

for t1 in close_prices:
    for t2 in close_prices:
        corr_mat.loc[t1, t2] = weighted_corr(t1, t2)

In [16]:
display(corr_mat)

,AAPL,ABBV,ABT,ACN,AIG,AMZN,AXP,BA,BAC,BB.TO,...,PYPL,QCOM,SHOP.TO,T.TO,TD.TO,TXN,UNH,UNP,UPS,USB
AAPL,1.000000,0.216744,0.113643,0.186809,0.241975,0.097348,0.248974,0.257245,0.258058,0.022674,...,0.264516,0.240358,0.244646,0.321750,0.212976,0.159325,-0.005418,0.116131,0.247728,0.281181
ABBV,0.216744,1.000000,0.356947,0.212643,0.204026,-0.110452,0.120386,0.023360,0.103718,-0.028655,...,0.147429,-0.017752,-0.001030,0.138242,0.018531,-0.038252,0.159852,0.234680,0.180004,0.179568
ABT,0.113643,0.356947,1.000000,0.283775,0.241838,-0.037806,0.135869,0.106955,0.154943,-0.220730,...,0.166938,-0.097804,0.145285,0.096818,-0.006697,-0.112216,0.113007,0.277915,0.164430,0.213050
ACN,0.186809,0.212643,0.283775,1.000000,0.232398,0.073419,0.282050,0.113523,0.119036,0.033997,...,0.294870,-0.004969,0.373909,0.288899,-0.053158,-0.240543,0.089481,0.082734,0.131467,0.223386
AIG,0.241975,0.204026,0.241838,0.232398,1.000000,0.031943,0.200183,0.171519,0.238466,-0.117127,...,0.074099,0.005842,0.108990,0.054798,0.132519,0.000187,0.126738,0.301400,0.143122,0.263614
AMZN,0.097348,-0.110452,-0.037806,0.073419,0.031943,1.000000,0.320773,0.281959,0.265232,0.148970,...,0.201944,0.182463,0.217566,-0.315896,0.228405,0.155232,0.051800,0.052965,0.148410,0.264464
AXP,0.248974,0.120386,0.135869,0.282050,0.200183,0.320773,1.000000,0.405369,0.606755,0.156115,...,0.308851,0.225856,0.365525,0.073358,0.428863,0.174560,0.221438,0.190186,0.317118,0.681290
BA,0.257245,0.023360,0.106955,0.113523,0.171519,0.281959,0.405369,1.000000,0.343083,0.050061,...,0.218346,0.295101,0.185358,0.041692,0.306210,0.282194,0.140874,0.098535,0.272391,0.340830
BAC,0.258058,0.103718,0.154943,0.119036,0.238466,0.265232,0.606755,0.343083,1.000000,0.081433,...,0.165916,0.217148,0.135901,-0.032668,0.507937,0.175587,0.140715,0.209081,0.288252,0.749104
BB.TO,0.022674,-0.028655,-0.220730,0.033997,-0.117127,0.148970,0.156115,0.050061,0.081433,1.000000,...,0.071481,0.217196,0.182644,-0.081699,0.222320,0.190345,0.052468,0.022913,0.147297,0.134091


In [17]:
period = 15
threshold = 1.5

In [18]:
# Note: this shouldn't be used as a raw value, as there may be periods where the variance
# jumps are spread across two periods and thus not counted. Use a threshold lower than
# what you're truly looking for. 
def count_jumps_std(period, threshold):
    result = pd.DataFrame()
    std_dict = {}
    for tick in close_prices:
        result.loc[tick, "Occurrences"] = 0
        std_dict[tick] = close_prices[tick].std()
    interval = [len(full_data)-period, len(full_data)-1]
    while interval[0] >= 0:
        for tick in close_prices:
            if full_data.iloc[interval][tick].std() >= std_dict[tick] * threshold:
                result.loc[tick, "Occurrences"]+=1
        interval[0]-=period
        interval[1]-=period

    return result

count_jumps_std(period, threshold)

,Occurrences
AAPL,0.0
ABBV,2.0
ABT,0.0
ACN,0.0
AIG,6.0
AMZN,5.0
AXP,2.0
BA,9.0
BAC,0.0
BB.TO,0.0


In [19]:
period = 15
threshold = 0.8

In [20]:
# Note: this shouldn't be used as a raw value, as there may be periods where the price
# jumps are spread across two periods and thus not counted. Use a threshold lower than
# what you're truly looking for. 

def count_price_plummets(period, threshold):
    result = pd.DataFrame()
    std_dict = {}
    for tick in close_prices:
        result.loc[tick, "Occurrences"] = 0
    interval = [len(full_data)-period, len(full_data)-1]
    while interval[0] >= 0:
        for tick in close_prices:
            if full_data.iloc[interval[1]][tick] <= full_data.iloc[interval[0]][tick] * threshold:
                result.loc[tick, "Occurrences"]+=1
        interval[0]-=period
        interval[1]-=period

    return result

count_price_plummets(period, threshold)

,Occurrences
AAPL,0.0
ABBV,0.0
ABT,0.0
ACN,1.0
AIG,0.0
AMZN,0.0
AXP,0.0
BA,0.0
BAC,0.0
BB.TO,2.0


In [39]:
sectors = {}

for tick in close_prices:
    sectors[tick] = yf.Ticker(tick).info.get("sector")

High growth

In [44]:
drop_threshold = 2

price_drops = count_price_plummets(period, threshold)

valid_ticks = []

for tick in close_prices:
    if price_drops["Occurrences"][tick] < drop_threshold:
        valid_ticks.append(tick)

valid_ticks = sorted(valid_ticks, key=lambda x: (-sharpe[x]))

sector_chosen = {}

chosen_stocks = 0

for tick in valid_ticks:
    sector = sectors[tick]

    if chosen_stocks >= 20:
        break
    
    if (sector in sector_chosen) and len(sector_chosen[sector]) == 3:
        continue
    else:
        if sector not in sector_chosen:
            sector_chosen[sector] = [tick]
        else:
            sector_chosen[sector].append(tick)

    chosen_stocks += 1

print(sector_chosen)

{'Financial Services': ['TD.TO', 'RY.TO', 'C'], 'Healthcare': ['MRK', 'ABBV', 'BIIB'], 'Technology': ['AAPL', 'TXN', 'SHOP.TO'], 'Consumer Defensive': ['KO', 'PM', 'MO'], 'Industrials': ['CAT', 'UNP', 'LMT'], 'Consumer Cyclical': ['AMZN'], 'Communication Services': ['T.TO']}
